# Forward Pass

Every layer does two things:
- Linear: Z = W·A_prev + B
- Nonlinear: A = g(Z)

We implement this manually in NumPy first (see every multiply) then vectorized in PyTorch.

In [1]:
# MANUAL FORWARD — NumPy (see every op)
import numpy as np
import torch.nn as nn
import torch

def sigmoid(z):                                           # squash any real to (0, 1)
    return 1 / (1 + np.exp(-z))
    # REASONING: sigmoid because binary output needs a probability.
    # Alternative: tanh maps to (−1,1) — used when you want zero-centered outputs.

def relu(z):                                              # max(0, z) element-wise
    return np.maximum(0, z)
    # REASONING: np.maximum is a single ufunc — faster than np.where(z>0, z, 0)
    # which allocates a boolean mask AND the output.

x = np.array([0.5, 1.0])                                  # input vector (2,)

# Layer 1: 2 → 3
W1 = np.array([[0.1, 0.2, 0.3],                           # (in=2, out=3)
               [0.4, 0.5, 0.6]])
# REASONING: we store W as (in, out) → matmul x @ W gives (out,).
# PyTorch stores W as (out, in) internally and computes x @ W.T.
# Either convention works; mixing them is the #1 shape bug.
B1 = np.array([0.1, 0.1, 0.1])                            # (out=3,)
Z1 = W1.T @ x + B1                                        # (3,2)ᵀ@(2,) + (3,) → (3,)
A1 = sigmoid(Z1)                                          # element-wise (3,)

# Layer 2: 3 → 1
W2 = np.array([[0.1], [0.2], [0.3]])                      # (in=3, out=1)
B2 = np.array([0.1])                                      # (out=1,)
Z2 = W2.T @ A1 + B2                                       # (1,)
A2 = sigmoid(Z2)                                          # final output
print(f"A2 (output) = {A2}")

A2 (output) = [0.62415485]


In [2]:
# VECTORIZED FORWARD — PyTorch
class SimpleMLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()                                # required
        self.layer1 = nn.Linear(input_size, hidden_size)  # in → hidden
        # REASONING: nn.Linear(in, out) stores W as (out, in) internally.
        # forward computes x @ W.T + b. Different from our NumPy convention —
        # this is the PyTorch convention and worth memorizing.
        self.relu = nn.ReLU()                             # nonlinear
        self.layer2 = nn.Linear(hidden_size, output_size) # hidden → out
        self.sigmoid = nn.Sigmoid()                       # output activation

    def forward(self, x):                                 # declare data flow
        z1 = self.layer1(x)                               # x @ W1ᵀ + b1
        a1 = self.relu(z1)                                # max(0, z1)
        z2 = self.layer2(a1)                              # a1 @ W2ᵀ + b2
        y_hat = self.sigmoid(z2)                          # squash to (0,1)
        return y_hat

model = SimpleMLP(2, 3, 1)
x = torch.tensor([[0.5, 1.0]], dtype=torch.float32)

model.eval()                                              # INFERENCE MODE
with torch.no_grad():                                     # no autograd graph
    y_hat = model(x)
print(f"Vectorized forward: {y_hat.item():.4f}")

Vectorized forward: 0.4203


#### Alternatives

In [3]:
# Alt 1: Sequential
model_seq = nn.Sequential(nn.Linear(2, 3), nn.ReLU(), nn.Linear(3, 1), nn.Sigmoid())
# WHY: 1 line, easy to read. LIMIT: no branching, no skip connections.

# Alt 2: Explicit Module
class MLPExplicit(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 3)
        self.fc2 = nn.Linear(3, 1)
    def forward(self, x):
        return torch.sigmoid(self.fc2(torch.relu(self.fc1(x))))
# WHY: verbose but clear. LIMIT: same as Sequential for simple cases.

# Alt 3: Functional
def functional_forward(params, x):
    x = torch.relu(x @ params['w1'].T + params['b1'])
    return torch.sigmoid(x @ params['w2'].T + params['b2'])
# WHY: no state → easy to vmap/grad. USE: research, JAX-style code.

# Alt 4: Logits only (no output activation) — needed for BCEWithLogitsLoss
class MLPLogits(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 3)
        self.fc2 = nn.Linear(3, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))          # raw logits
# WHY: BCEWithLogitsLoss fuses sigmoid+BCE for numerical stability.
# If you apply sigmoid here AND then use BCELoss, you can get log(0) → NaN.

#### What could go wrong?

In [4]:
# FAILURE 1: Wrong W orientation → shape error
# REASONING: (in, out) vs (out, in) are both valid conventions, but mixing
# them gives `matmul: Input operand 1 has a mismatch` if dims don't align,
# or SILENTLY WRONG results if they happen to align by coincidence.
x = np.array([0.5, 1.0])
W1_bad = np.random.randn(3, 2)                            # should be (2, 3)
try:
    _ = W1_bad.T @ x
except ValueError as e:
    print("Shape error:", e)

Shape error: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)


In [5]:
# FAILURE 2: Naive sigmoid overflows for extreme z
# REASONING: exp(1000) = inf. The stable form splits into pos/neg branches
# so the exponent is always ≤ 0.
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(1 / (1 + np.exp(-np.array([-1000., 0., 1000.]))))

[0.  0.5 1. ]


In [6]:
# FAILURE 3: Forgot .eval() → dropout randomness at inference
# REASONING: nn.Dropout scales activations by 1/(1-p) and randomly zeros
# p·N of them in train mode. At eval it's identity. Forgetting .eval()
# means your "test" predictions vary run-to-run — silently wrong.
m = nn.Sequential(nn.Linear(4, 8), nn.Dropout(0.5), nn.Linear(8, 1))
m.train()                                                 # TRAIN MODE
x_ = torch.randn(1, 4)
print("train differ:", not torch.allclose(m(x_), m(x_)))  # True
m.eval()                                                  # INFERENCE MODE
print("eval same:  ", torch.allclose(m(x_), m(x_)))       # True

train differ: True
eval same:   True


In [7]:
# FAILURE 4: BatchNorm with batch_size=1 → error
# REASONING: BatchNorm computes per-feature mean/var over the batch dimension.
# With batch size 1, variance = 0 → division by sqrt(0+eps) = sqrt(eps) ≈ 0 → unstable.
# PyTorch raises: "Expected more than 1 value per channel when training".
try:
    m = nn.Sequential(nn.Linear(4, 8), nn.BatchNorm1d(8))
    m.train()
    _ = m(torch.randn(1, 4))
except Exception as e:
    print("BN error:", str(e)[:80])

BN error: Expected more than 1 value per channel when training, got input size torch.Size(
